# Process Awinda: IMU IK → ID vs OpenSim ID

Runs checkpoint inference from IMU IK and compares against OpenSim inverse dynamics for **awinda** (no-exo) trials.

- **IMU IK**: `mt_processed/{Subject}/ik/VQF/{speed}_{cond}.pkl` (cond is lowercase, e.g. `1p2mps_lg`)
- **OpenSim ID**: `processed/{Subject}/awinda/id/{COND}_{speed}_id.sto`
- **Vicon IK** (sync reference): `processed/{Subject}/awinda/ik/{COND}_{speed}_ik.mot`
- **Sync**: Awinda vs Vicon IK **angle xcorr** (hip+knee, z-scored, 5 s skip) → apply lag to model ID vs OpenSim ID, then **15 s transient trim**
- **Filters**: match training config (`zero_phase` 6 Hz angle/output, 15 Hz velocity)
- **Angle offsets** (IMU IK, before filter/inference/sync): `AB02_Oscar::LG_0p8mps` +6° all joints; `AB01_Jinwoo::RD_0p8mps` +10° knee+ankle
- **Output**: cached results in `analysis/cache/process_awinda.npz` — open `visualize_awinda.ipynb` to explore

Use kernel `jinwoo-addbiomech` (needs PyTorch).


In [1]:
import json
import pickle
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from scipy.signal import butter, sosfilt, sosfiltfilt, find_peaks

warnings.filterwarnings('ignore', message='.*NumPy.*')

PROJECT_ROOT = Path('/home/metamobility3/Jinwoo/os_kinetics').resolve()
PROCESSED_ROOT = Path('/media/metamobility3/Samsung_T52/Results/processed')
IMU_IK_ROOT = Path('/home/metamobility3/Jinwoo/mt_processed')
IMU_IK_METHOD = 'VQF'
CHECKPOINT = PROJECT_ROOT / 'runs/0512_ik_id_all_zero_in_zero_out/best_model.pt'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

SUBJECT_MASS_KG = {
    'AB01_Jinwoo': 88.0, 'AB02_Oscar': 71.1, 'AB03_Ilseung': 84.4,
    'AB04_Changseob': 74.0, 'AB05_Maria': 55.0, 'AB06_Jimin': 82.6,
    'AB07_Amy': 51.3, 'AB08_Seokhyun': 71.9,
}

CHANNELS = [
    'hip_flexion_r', 'knee_angle_r', 'ankle_angle_r',
    'hip_flexion_l', 'knee_angle_l', 'ankle_angle_l',
]
ID_COLS = [f'{c}_moment' for c in CHANNELS]
DEFAULT_FS_HZ = 100.0
ALIGN_MAX_LAG_SEC = 30.0
XCORR_SKIP_SEC = 5.0
TRANSIENT_TRIM_SEC = 15.0
PREVIEW_FRAMES = 500  # first 5 s at 100 Hz

# IMU IK constant offsets (deg) applied before LPF / inference / sync.
ANGLE_OFFSET_DEG = {
    ('AB02_Oscar', 'LG_0p8mps'): {c: 6.0 for c in CHANNELS},
    ('AB01_Jinwoo', 'RD_0p8mps'): {
        'knee_angle_r': 10.0, 'knee_angle_l': 10.0,
        'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0,
    },
}

# Still report under QC thresholds, but do not exclude from FILTERED_TRIAL_DATA.
QC_FORCE_KEEP = {
    'AB02_Oscar::RA_0p8mps',   # underperforming, not a failure
    'AB08_Seokhyun::RA_0p8mps',  # underperforming, not a failure
}

sys.path.insert(0, str(PROJECT_ROOT))
from dataset import IK_DOF_NAMES
from model import TCN

IK_CHANNEL_IDX = [IK_DOF_NAMES.index(c) for c in CHANNELS]
XCORR_CHANNEL_IDX = [CHANNELS.index(c) for c in ('hip_flexion_r', 'knee_angle_r', 'hip_flexion_l', 'knee_angle_l')]

print(f'Checkpoint: {CHECKPOINT}')
print(f'Processed root: {PROCESSED_ROOT}')
print(f'Device: {DEVICE}')
print(f'ANGLE_OFFSET_DEG: { {f"{s}::{c}": v for (s, c), v in ANGLE_OFFSET_DEG.items()} }')
print(f'QC_FORCE_KEEP: {sorted(QC_FORCE_KEEP)}')


Checkpoint: /home/metamobility3/Jinwoo/os_kinetics/runs/0512_ik_id_all_zero_in_zero_out/best_model.pt
Processed root: /media/metamobility3/Samsung_T52/Results/processed
Device: cuda
ANGLE_OFFSET_DEG: {'AB02_Oscar::LG_0p8mps': {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0, 'ankle_angle_r': 6.0, 'hip_flexion_l': 6.0, 'knee_angle_l': 6.0, 'ankle_angle_l': 6.0}, 'AB01_Jinwoo::RD_0p8mps': {'knee_angle_r': 10.0, 'knee_angle_l': 10.0, 'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0}}
QC_FORCE_KEEP: ['AB02_Oscar::RA_0p8mps', 'AB08_Seokhyun::RA_0p8mps']


In [2]:
def parse_opensim_table(path: Path) -> pd.DataFrame:
    with open(path) as f:
        header_end = next(i for i, line in enumerate(f) if line.strip().lower() == 'endheader')
    return pd.read_csv(path, sep=r'\s+', skiprows=header_end + 1).set_index('time')


def butter_lpf(x, fs_hz, cutoff_hz, order, mode='zero_phase'):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    nyq = 0.5 * fs_hz
    if cutoff_hz <= 0 or cutoff_hz >= nyq or len(x) < 4:
        return x.copy()
    sos = butter(order, cutoff_hz / nyq, btype='low', output='sos')
    return sosfiltfilt(sos, x) if mode == 'zero_phase' else sosfilt(sos, x)


def lpf_mc(X, fs_hz, cutoff_hz, order, mode='zero_phase'):
    return np.column_stack([butter_lpf(X[:, c], fs_hz, cutoff_hz, order, mode) for c in range(X.shape[1])])


def filename_to_condition(stem: str) -> str:
    speed, cond = stem.split('_', 1)
    return f'{cond.upper()}_{speed}'


def condition_to_pkl_stem(condition: str) -> str:
    cond, speed = condition.split('_', 1)
    return f'{speed}_{cond.lower()}'


def resolve_trial_paths(subject: str, condition: str):
    pkl_stem = condition_to_pkl_stem(condition)
    ik_dir = IMU_IK_ROOT / subject / 'ik' / IMU_IK_METHOD
    if not ik_dir.exists():
        ik_dir = IMU_IK_ROOT / subject
    pkl_path = ik_dir / f'{pkl_stem}.pkl'
    id_path = PROCESSED_ROOT / subject / 'awinda' / 'id' / f'{condition}_id.sto'
    vicon_path = PROCESSED_ROOT / subject / 'awinda' / 'ik' / f'{condition}_ik.mot'
    return {
        'subject': subject,
        'condition': condition,
        'pkl_path': pkl_path,
        'id_path': id_path,
        'vicon_path': vicon_path,
        'mass_kg': SUBJECT_MASS_KG.get(subject, np.nan),
    }


def list_available_trials():
    rows = []
    WARNINGS = []
    for subj_dir in sorted(IMU_IK_ROOT.glob('AB*')):
        if not subj_dir.is_dir():
            continue
        subject = subj_dir.name
        if not (PROCESSED_ROOT / subject).is_dir():
            WARNINGS.append(f'[WARN] No processed folder for {subject} — skipping all trials')
            continue
        ik_dir = subj_dir / 'ik' / IMU_IK_METHOD
        if not ik_dir.exists():
            ik_dir = subj_dir
        for pkl_path in sorted(ik_dir.glob('*.pkl')):
            cond = filename_to_condition(pkl_path.stem)
            id_path = PROCESSED_ROOT / subject / 'awinda' / 'id' / f'{cond}_id.sto'
            vicon_path = PROCESSED_ROOT / subject / 'awinda' / 'ik' / f'{cond}_ik.mot'
            ok = id_path.exists() and pkl_path.exists() and vicon_path.exists()
            if not ok:
                WARNINGS.append(f'[WARN] {subject}::{cond} missing id/vicon/pkl — skipped')
            rows.append({
                'subject': subject,
                'condition': cond,
                'pkl_path': pkl_path,
                'id_path': id_path if id_path.exists() else None,
                'vicon_path': vicon_path if vicon_path.exists() else None,
                'mass_kg': SUBJECT_MASS_KG.get(subject, np.nan),
                'ready': ok,
            })
    return pd.DataFrame(rows), WARNINGS


def clip_lag(lag, max_lag_samples):
    clipped = False
    if lag > max_lag_samples:
        lag, clipped = max_lag_samples, True
    elif lag < -max_lag_samples:
        lag, clipped = -max_lag_samples, True
    return int(lag), clipped


def apply_angle_offset_rad(pos_rad: np.ndarray, subject: str, condition: str) -> np.ndarray:
    offsets = ANGLE_OFFSET_DEG.get((subject, condition))
    if not offsets:
        return pos_rad
    out = np.asarray(pos_rad, dtype=np.float64).copy()
    for name, deg in offsets.items():
        out[:, IK_DOF_NAMES.index(name)] += np.deg2rad(float(deg))
    return out


def build_awinda_ik_rad(imu_dict: dict) -> np.ndarray:
    return np.deg2rad(build_model_input_from_pkl(imu_dict))


def build_vicon_ik_rad(ik_df: pd.DataFrame) -> np.ndarray:
    pos_deg = np.full((len(ik_df), len(IK_DOF_NAMES)), np.nan)
    for j, name in enumerate(IK_DOF_NAMES):
        if name in ik_df.columns:
            pos_deg[:, j] = ik_df[name].to_numpy(dtype=np.float64)
    return np.deg2rad(pos_deg)


def _zscore_1d(x):
    x = np.asarray(x, dtype=np.float64)
    m = np.isfinite(x)
    if m.sum() < 2:
        return np.zeros_like(x)
    mu, sd = float(np.nanmean(x[m])), float(np.nanstd(x[m]))
    if sd < 1e-9:
        return np.zeros_like(x)
    out = (x - mu) / sd
    out[~m] = 0.0
    return out


def estimate_lag_from_angle_xcorr(
    awinda_rad,
    vicon_rad,
    fs_hz,
    max_lag_samples,
    channel_idx=None,
    skip_s=XCORR_SKIP_SEC,
    angle_cutoff=6.0,
    filter_order=4,
    in_mode='zero_phase',
):
    """Multi-channel z-scored cross-correlation. lag = awinda_idx - vicon_idx."""
    channel_idx = list(channel_idx or XCORR_CHANNEL_IDX)
    awinda_f = lpf_mc(awinda_rad[:, IK_CHANNEL_IDX], fs_hz, angle_cutoff, filter_order, in_mode)
    vicon_f = lpf_mc(vicon_rad[:, IK_CHANNEL_IDX], fs_hz, angle_cutoff, filter_order, in_mode)
    skip = int(round(skip_s * fs_hz))
    best_lag, best_score = 0, -np.inf
    for lag in range(-max_lag_samples, max_lag_samples + 1):
        start_a, start_b = max(lag, 0), max(-lag, 0)
        nn = min(len(awinda_f) - start_a, len(vicon_f) - start_b) - skip
        if nn < int(round(2.0 * fs_hz)):
            continue
        seg_a = awinda_f[start_a + skip:start_a + skip + nn]
        seg_b = vicon_f[start_b + skip:start_b + skip + nn]
        score = sum(float(np.dot(_zscore_1d(seg_a[:, c]), _zscore_1d(seg_b[:, c])) / nn) for c in channel_idx)
        if score > best_score:
            best_score, best_lag = score, lag
    lag, clipped = clip_lag(best_lag, max_lag_samples)
    return lag, {'xcorr_score': float(best_score), 'lag_clipped': clipped}


def build_model_input_from_pkl(imu_dict: dict) -> np.ndarray:
    n = len(next(iter(imu_dict.values())))
    pos_deg = np.zeros((n, len(IK_DOF_NAMES)), dtype=np.float64)
    key_map = {
        'hip_flexion_r': 'hip_flexion_r', 'knee_angle_r': 'knee_flexion_r', 'ankle_angle_r': 'ankle_flexion_r',
        'hip_flexion_l': 'hip_flexion_l', 'knee_angle_l': 'knee_flexion_l', 'ankle_angle_l': 'ankle_flexion_l',
    }
    sign_map = {'knee_angle_r': -1.0, 'knee_angle_l': -1.0}
    for ik_name, pkl_name in key_map.items():
        idx = IK_DOF_NAMES.index(ik_name)
        pos_deg[:, idx] = sign_map.get(ik_name, 1.0) * np.asarray(imu_dict[pkl_name], dtype=np.float64)
    return pos_deg


def rmse_r2(y_pred, y_true):
    """RMSE and coefficient of determination R² = 1 - SSE/SST (matches exo analyses)."""
    m = np.isfinite(y_pred) & np.isfinite(y_true)
    if m.sum() < 2:
        return np.nan, np.nan
    e = y_pred[m] - y_true[m]
    rmse = float(np.sqrt(np.mean(e ** 2)))
    ss_res = float(np.sum(e ** 2))
    ss_tot = float(np.sum((y_true[m] - np.mean(y_true[m])) ** 2))
    r2 = float(1.0 - ss_res / (ss_tot + 1e-12))
    return rmse, r2


print('Helpers ready.')


Helpers ready.


In [3]:
ckpt = torch.load(str(CHECKPOINT), map_location=DEVICE, weights_only=False)
with open(CHECKPOINT.parent / 'config.json') as f:
    train_cfg = json.load(f)

cfg = ckpt['model_config']
model = TCN(**{k: cfg[k] for k in ['n_input_channels', 'n_output_channels', 'hidden_channels', 'n_blocks', 'kernel_size', 'dropout']})
model.load_state_dict(ckpt['model_state_dict'])
model.to(DEVICE).eval()

WINDOW_SIZE = int(ckpt.get('window_size', 100))
INPUT_INDICES = list(ckpt.get('input_indices', [6, 9, 10, 13, 16, 17]))
H = len(INPUT_INDICES) // 2
INPUT_IDX_R, INPUT_IDX_L = INPUT_INDICES[:H], INPUT_INDICES[H:]

ANGLE_CUTOFF = float(train_cfg.get('lowpass_cutoff_hz', 6.0))
VEL_CUTOFF = float(train_cfg.get('velocity_lowpass_cutoff_hz', 15.0))
OUT_CUTOFF = float(train_cfg.get('lowpass_cutoff_hz', 6.0))
FILTER_ORDER = int(train_cfg.get('lowpass_order', 4))
IN_MODE = str(train_cfg.get('input_lowpass_mode', 'zero_phase'))
OUT_MODE = str(train_cfg.get('output_lowpass_mode', 'zero_phase'))

@torch.no_grad()
def infer_one_side(pos_3, vel_3):
    x = np.concatenate([pos_3, vel_3], axis=1).astype(np.float32)
    n, W, c_out = x.shape[0], WINDOW_SIZE, cfg['n_output_channels']
    pred = np.zeros((n, c_out), dtype=np.float64)
    def _fwd(start):
        xt = torch.from_numpy(np.ascontiguousarray(x[start:start + W].T)).unsqueeze(0).to(DEVICE)
        return model(xt).squeeze(0).detach().cpu().numpy().T
    pred[:W] = _fwd(0)
    for start in range(1, n - W + 1):
        pred[start + W - 1] = _fwd(start)[W - 1]
    return pred.astype(np.float32)


def run_bilateral_inference(pos_full, vel_full):
    pr = infer_one_side(pos_full[:, INPUT_IDX_R], vel_full[:, INPUT_IDX_R])
    pl = infer_one_side(pos_full[:, INPUT_IDX_L], vel_full[:, INPUT_IDX_L])
    return np.concatenate([pr, pl], axis=1)

print(f'window={WINDOW_SIZE}, filters: angle={ANGLE_CUTOFF}Hz/{IN_MODE}, vel={VEL_CUTOFF}Hz/{IN_MODE}, out={OUT_CUTOFF}Hz/{OUT_MODE}')

window=100, filters: angle=6.0Hz/zero_phase, vel=15.0Hz/zero_phase, out=6.0Hz/zero_phase


In [4]:
manifest, WARNINGS = list_available_trials()
for w in WARNINGS:
    print(w)
print(f'\nDiscovered {len(manifest)} IMU trials | ready={int(manifest["ready"].sum())} | skipped={int((~manifest["ready"]).sum())}')
display(manifest[['subject', 'condition', 'ready', 'mass_kg']])



Discovered 40 IMU trials | ready=40 | skipped=0


,subject,condition,ready,mass_kg
0,AB01_Jinwoo,LG_0p8mps,True,88.0
1,AB01_Jinwoo,RA_0p8mps,True,88.0
2,AB01_Jinwoo,RD_0p8mps,True,88.0
3,AB01_Jinwoo,LG_1p2mps,True,88.0
4,AB01_Jinwoo,LG_1p6mps,True,88.0
5,AB02_Oscar,LG_0p8mps,True,71.1
6,AB02_Oscar,RA_0p8mps,True,71.1
7,AB02_Oscar,RD_0p8mps,True,71.1
8,AB02_Oscar,LG_1p2mps,True,71.1
9,AB02_Oscar,LG_1p6mps,True,71.1


In [5]:
TRIAL_DATA = {}
rows = []
ANKLE_MOMENT_IDX = CHANNELS.index('ankle_angle_r')

for _, row in manifest[manifest['ready']].iterrows():
    subject, cond = row['subject'], row['condition']
    trial_key = f'{subject}::{cond}'
    try:
        imu = pickle.load(open(row['pkl_path'], 'rb'))
        pos_rad = apply_angle_offset_rad(
            np.deg2rad(build_model_input_from_pkl(imu)), subject, cond,
        )
        id_df = parse_opensim_table(row['id_path'])
        t_id = id_df.index.to_numpy(dtype=np.float64)
        fs_hz = 1.0 / float(np.median(np.diff(t_id))) if len(t_id) > 2 else DEFAULT_FS_HZ

        pos_f = lpf_mc(pos_rad, fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)
        vel_f = lpf_mc(np.gradient(pos_f, 1.0 / fs_hz, axis=0), fs_hz, VEL_CUTOFF, FILTER_ORDER, IN_MODE)
        pred_nmpkg_full = run_bilateral_inference(pos_f, vel_f)
        pred_nmpkg_full_f = lpf_mc(pred_nmpkg_full, fs_hz, OUT_CUTOFF, FILTER_ORDER, OUT_MODE)

        id_nm = np.column_stack([
            id_df[c].to_numpy(dtype=np.float64) if c in id_df.columns else np.full(len(t_id), np.nan)
            for c in ID_COLS
        ])
        id_nmpkg_full_f = lpf_mc(id_nm / row['mass_kg'], fs_hz, OUT_CUTOFF, FILTER_ORDER, OUT_MODE)

        awinda_rad = pos_rad  # same IMU IK (incl. any ANGLE_OFFSET_DEG)
        vicon_rad = build_vicon_ik_rad(parse_opensim_table(row['vicon_path']))
        if (subject, cond) in ANGLE_OFFSET_DEG:
            print(f'ANGLE OFFSET {trial_key}: {ANGLE_OFFSET_DEG[(subject, cond)]}')
        max_lag = int(round(ALIGN_MAX_LAG_SEC * fs_hz))
        lag, sync_info = estimate_lag_from_angle_xcorr(
            awinda_rad,
            vicon_rad,
            fs_hz,
            max_lag,
            skip_s=XCORR_SKIP_SEC,
            angle_cutoff=ANGLE_CUTOFF,
            filter_order=FILTER_ORDER,
            in_mode=IN_MODE,
        )

        start_pred, start_id = max(lag, 0), max(-lag, 0)
        n_sync = min(len(pred_nmpkg_full_f) - start_pred, len(id_nmpkg_full_f) - start_id)
        trim_n = int(round(TRANSIENT_TRIM_SEC * fs_hz))
        if n_sync - trim_n < WINDOW_SIZE:
            raise RuntimeError(f'Synced window too short after {TRANSIENT_TRIM_SEC}s trim: {n_sync - trim_n}')

        pred_nmpkg_f = pred_nmpkg_full_f[start_pred + trim_n:start_pred + n_sync]
        id_nmpkg_f = id_nmpkg_full_f[start_id + trim_n:start_id + n_sync]
        t_sync = t_id[start_id + trim_n:start_id + n_sync]

        metrics = []
        for c in range(6):
            rmse, r2 = rmse_r2(pred_nmpkg_f[:, c], id_nmpkg_f[:, c])
            metrics.append({'channel': CHANNELS[c], 'rmse_nmpkg': rmse, 'r2_nmpkg': r2})

        n_pre = min(PREVIEW_FRAMES, len(pred_nmpkg_full_f), len(id_nmpkg_full_f))
        awinda_hip = np.rad2deg(lpf_mc(awinda_rad[:, IK_CHANNEL_IDX], fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)[:n_pre, 0])
        vicon_hip = np.rad2deg(lpf_mc(vicon_rad[:, IK_CHANNEL_IDX], fs_hz, ANGLE_CUTOFF, FILTER_ORDER, IN_MODE)[:n_pre, 0])
        sync_debug = {
            'fs_hz': fs_hz,
            'sync_method': 'angle_xcorr',
            'xcorr_score': sync_info['xcorr_score'],
            't_id': t_id[:n_pre],
            't_imu': np.arange(n_pre) / fs_hz,
            'pred_pre': pred_nmpkg_full_f[:n_pre],
            'id_pre': id_nmpkg_full_f[:n_pre],
            'pred_ankle': pred_nmpkg_full_f[:n_pre, ANKLE_MOMENT_IDX],
            'id_ankle': id_nmpkg_full_f[:n_pre, ANKLE_MOMENT_IDX],
            'awinda_hip_deg': awinda_hip,
            'vicon_hip_deg': vicon_hip,
            'n_sync': n_sync,
            'trim_n': trim_n,
            'start_pred': start_pred,
            'start_id': start_id,
            **sync_info,
        }

        TRIAL_DATA[trial_key] = {
            'subject': subject, 'condition': cond, 't': t_sync, 'mass_kg': row['mass_kg'],
            'pred_nmpkg': pred_nmpkg_f, 'id_nmpkg': id_nmpkg_f, 'metrics': metrics,
            'lag_samples': lag, 'lag_seconds': lag / fs_hz, 'sync_debug': sync_debug,
            'xcorr_score': sync_info['xcorr_score'], 'lag_clipped': sync_info['lag_clipped'],
        }
        mean_rmse = np.nanmean([m['rmse_nmpkg'] for m in metrics])
        mean_r2 = np.nanmean([m['r2_nmpkg'] for m in metrics])
        rows.append({
            'trial': trial_key, 'n': len(pred_nmpkg_f),
            'mean_rmse': mean_rmse,
            'mean_r2': mean_r2,
        })
        print(
            f'OK  {trial_key} | lag={lag:+d} samples ({lag / fs_hz:+.3f} s) | '
            f'xcorr={sync_info["xcorr_score"]:.3f} clipped={sync_info["lag_clipped"]} | '
            f'mean RMSE={mean_rmse:.4f} R²={mean_r2:.4f}'
        )
    except Exception as exc:
        WARNINGS.append(f'[WARN] Failed {trial_key}: {exc}')
        print(f'FAIL {trial_key}: {exc}')

summary_df = pd.DataFrame(rows)
detail_rows = []
for trial, d in TRIAL_DATA.items():
    for m in d['metrics']:
        detail_rows.append({'trial': trial, **m})
detail_df = pd.DataFrame(detail_rows)

print(f'\nLoaded {len(TRIAL_DATA)} trials')
if not detail_df.empty:
    print('\nPer-joint mean across trials:')
    display(detail_df.groupby('channel')[['rmse_nmpkg', 'r2_nmpkg']].mean())
    print('\nOverall (all trials × joints):')
    print(f"  RMSE = {detail_df['rmse_nmpkg'].mean():.4f} N·m/kg")
    print(f"  R²   = {detail_df['r2_nmpkg'].mean():.4f}")


OK  AB01_Jinwoo::LG_0p8mps | lag=+711 samples (+7.110 s) | xcorr=3.946 clipped=False | mean RMSE=0.1945 R²=0.2980
OK  AB01_Jinwoo::RA_0p8mps | lag=+873 samples (+8.730 s) | xcorr=3.976 clipped=False | mean RMSE=0.1666 R²=0.7133
ANGLE OFFSET AB01_Jinwoo::RD_0p8mps: {'knee_angle_r': 10.0, 'knee_angle_l': 10.0, 'ankle_angle_r': 10.0, 'ankle_angle_l': 10.0}
OK  AB01_Jinwoo::RD_0p8mps | lag=+746 samples (+7.460 s) | xcorr=3.918 clipped=False | mean RMSE=0.2372 R²=0.3998
OK  AB01_Jinwoo::LG_1p2mps | lag=+722 samples (+7.220 s) | xcorr=3.943 clipped=False | mean RMSE=0.1816 R²=0.6748
OK  AB01_Jinwoo::LG_1p6mps | lag=+1300 samples (+13.000 s) | xcorr=3.938 clipped=False | mean RMSE=0.1823 R²=0.8069
ANGLE OFFSET AB02_Oscar::LG_0p8mps: {'hip_flexion_r': 6.0, 'knee_angle_r': 6.0, 'ankle_angle_r': 6.0, 'hip_flexion_l': 6.0, 'knee_angle_l': 6.0, 'ankle_angle_l': 6.0}
OK  AB02_Oscar::LG_0p8mps | lag=+386 samples (+3.860 s) | xcorr=3.947 clipped=False | mean RMSE=0.1670 R²=-0.4204
OK  AB02_Oscar::RA_

,rmse_nmpkg,r2_nmpkg
channel,,
ankle_angle_l,0.175127,0.862524
ankle_angle_r,0.164767,0.882227
hip_flexion_l,0.150386,0.742366
hip_flexion_r,0.155496,0.718169
knee_angle_l,0.152617,0.498195
knee_angle_r,0.125497,0.653151



Overall (all trials × joints):
  RMSE = 0.1540 N·m/kg
  R²   = 0.7261


In [6]:
CACHE_DIR = PROJECT_ROOT / 'analysis' / 'cache'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
CACHE_PATH = CACHE_DIR / 'process_awinda.npz'


def _trial_key_to_prefix(trial_key: str) -> str:
    return trial_key.replace('::', '__')


def save_processed_awinda_cache(trial_data, summary_df=None, detail_df=None, path=CACHE_PATH):
    if not trial_data:
        raise RuntimeError('TRIAL_DATA is empty — run processing cell first.')
    payload = {
        'channels': np.array(CHANNELS),
        'trial_keys': np.array(sorted(trial_data.keys()), dtype=object),
        'sync_method': np.array('angle_xcorr'),
        'xcorr_skip_sec': np.array(XCORR_SKIP_SEC),
        'transient_trim_sec': np.array(TRANSIENT_TRIM_SEC),
        'preview_frames': np.array(PREVIEW_FRAMES),
    }
    if summary_df is not None and not summary_df.empty:
        payload['summary_df'] = np.array(summary_df.to_json(orient='split'), dtype=object)
    if detail_df is not None and not detail_df.empty:
        payload['detail_df'] = np.array(detail_df.to_json(orient='split'), dtype=object)

    for trial_key, d in trial_data.items():
        p = _trial_key_to_prefix(trial_key)
        payload[f'{p}__t'] = np.asarray(d['t'], dtype=np.float64)
        payload[f'{p}__pred_nmpkg'] = np.asarray(d['pred_nmpkg'], dtype=np.float32)
        payload[f'{p}__id_nmpkg'] = np.asarray(d['id_nmpkg'], dtype=np.float32)
        payload[f'{p}__meta'] = np.array([
            d['subject'], d['condition'], d['mass_kg'],
            d['lag_samples'], d['lag_seconds'],
            d['xcorr_score'], d['lag_clipped'],
        ], dtype=object)
        rmse = [m['rmse_nmpkg'] for m in d['metrics']]
        r2 = [m['r2_nmpkg'] for m in d['metrics']]
        payload[f'{p}__rmse_nmpkg'] = np.asarray(rmse, dtype=np.float64)
        payload[f'{p}__r2_nmpkg'] = np.asarray(r2, dtype=np.float64)

        sd = d['sync_debug']
        payload[f'{p}__sync_fs_hz'] = np.array(sd['fs_hz'])
        payload[f'{p}__sync_n'] = np.array(sd['n_sync'])
        payload[f'{p}__sync_t_id'] = np.asarray(sd['t_id'], dtype=np.float64)
        payload[f'{p}__sync_t_imu'] = np.asarray(sd['t_imu'], dtype=np.float64)
        payload[f'{p}__sync_pred_pre'] = np.asarray(sd['pred_pre'], dtype=np.float32)
        payload[f'{p}__sync_id_pre'] = np.asarray(sd['id_pre'], dtype=np.float32)
        payload[f'{p}__sync_pred_ankle'] = np.asarray(sd['pred_ankle'], dtype=np.float32)
        payload[f'{p}__sync_id_ankle'] = np.asarray(sd['id_ankle'], dtype=np.float32)
        if 'awinda_hip_deg' in sd:
            payload[f'{p}__sync_awinda_hip_deg'] = np.asarray(sd['awinda_hip_deg'], dtype=np.float32)
            payload[f'{p}__sync_vicon_hip_deg'] = np.asarray(sd['vicon_hip_deg'], dtype=np.float32)
        if 'xcorr_score' in sd:
            payload[f'{p}__sync_xcorr_score'] = np.array(sd['xcorr_score'])

    np.savez_compressed(str(path), **payload)
    print(f'Saved {len(trial_data)} trials → {path}')


save_processed_awinda_cache(TRIAL_DATA, summary_df, detail_df)


Saved 40 trials → /home/metamobility3/Jinwoo/os_kinetics/analysis/cache/process_awinda.npz


In [7]:
MIN_R2 = 0.7
MAX_RMSE = 0.2

if summary_df.empty:
    raise RuntimeError('No processed trials — run the processing cell first.')

below = summary_df[
    (summary_df['mean_r2'] < MIN_R2) | (summary_df['mean_rmse'] > MAX_RMSE)
].copy()
force_kept = below[below['trial'].isin(QC_FORCE_KEEP)].copy()
flagged = below[~below['trial'].isin(QC_FORCE_KEEP)].copy()
flagged[['subject', 'condition']] = flagged['trial'].str.split('::', n=1, expand=True)
flagged['fail_r2'] = flagged['mean_r2'] < MIN_R2
flagged['fail_rmse'] = flagged['mean_rmse'] > MAX_RMSE

kept = summary_df[~summary_df['trial'].isin(flagged['trial'])].copy()

print(
    f'QC filter: mean R² < {MIN_R2} or mean RMSE > {MAX_RMSE} N·m/kg\n'
    f'  below threshold: {len(below)} / {len(summary_df)} trials\n'
    f'  force-kept (not excluded): {len(force_kept)}\n'
    f'  flagged/excluded: {len(flagged)}\n'
    f'  kept:    {len(kept)} trials'
)
if not force_kept.empty:
    print('  force-kept trials:', ', '.join(sorted(force_kept['trial'])))

if flagged.empty:
    print('\nNo trials flagged.')
else:
    print('\nFlagged trials by subject:')
    for subject in sorted(flagged['subject'].unique()):
        sub = flagged[flagged['subject'] == subject].sort_values('condition')
        trials = []
        for _, row in sub.iterrows():
            reasons = []
            if row['fail_r2']:
                reasons.append(f"R²={row['mean_r2']:.3f}")
            if row['fail_rmse']:
                reasons.append(f"RMSE={row['mean_rmse']:.3f}")
            trials.append(f"{row['condition']} ({', '.join(reasons)})")
        print(f'  {subject}: {", ".join(trials)}')

    display(
        flagged.sort_values(['subject', 'condition'])[
            ['subject', 'condition', 'mean_r2', 'mean_rmse', 'n', 'fail_r2', 'fail_rmse']
        ].reset_index(drop=True)
    )

EXCLUDED_TRIALS = set(flagged['trial'])
FILTERED_TRIAL_DATA = {k: v for k, v in TRIAL_DATA.items() if k not in EXCLUDED_TRIALS}
print(f'\nFILTERED_TRIAL_DATA: {len(FILTERED_TRIAL_DATA)} trials (excluded {len(EXCLUDED_TRIALS)})')

QC filter: mean R² < 0.7 or mean RMSE > 0.2 N·m/kg
  below threshold: 10 / 40 trials
  force-kept (not excluded): 2
  flagged/excluded: 8
  kept:    32 trials
  force-kept trials: AB02_Oscar::RA_0p8mps, AB08_Seokhyun::RA_0p8mps

Flagged trials by subject:
  AB01_Jinwoo: LG_0p8mps (R²=0.298), LG_1p2mps (R²=0.675), RD_0p8mps (R²=0.400, RMSE=0.237)
  AB02_Oscar: LG_0p8mps (R²=-0.420), RD_0p8mps (R²=0.680)
  AB04_Changseob: LG_0p8mps (R²=0.651), RD_0p8mps (R²=0.674)
  AB05_Maria: LG_0p8mps (R²=0.105, RMSE=0.429)


,subject,condition,mean_r2,mean_rmse,n,fail_r2,fail_rmse
0,AB01_Jinwoo,LG_0p8mps,0.298048,0.194493,6500,True,False
1,AB01_Jinwoo,LG_1p2mps,0.674819,0.181588,6500,True,False
2,AB01_Jinwoo,RD_0p8mps,0.399768,0.237238,6500,True,True
3,AB02_Oscar,LG_0p8mps,-0.420366,0.167042,6500,True,False
4,AB02_Oscar,RD_0p8mps,0.679823,0.181409,6500,True,False
5,AB04_Changseob,LG_0p8mps,0.650931,0.129210,6500,True,False
6,AB04_Changseob,RD_0p8mps,0.673811,0.159458,6500,True,False
7,AB05_Maria,LG_0p8mps,0.104919,0.428935,6500,True,True



FILTERED_TRIAL_DATA: 32 trials (excluded 8)
